<a href="https://colab.research.google.com/github/yoginikumar0608-gh/GenZSpace/blob/main/AI_Weather_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -qU langchain langchain-core langchain-google-genai langchain-tavily langgraph langsmith gradio

In [15]:
import os
import getpass

print("🔐 AI Health Information Researcher")
print("=" * 60)

os.environ["GEMINI_API_KEY"] = getpass.getpass(
    "Enter your Gemini API key: "
)

os.environ["TAVILY_API_KEY"] = getpass.getpass(
    "Enter your Tavily API key: "
)

os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
    "Enter your LangSmith API key: "
)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI-Health-Information-Researcher"

print("\n✅ API keys loaded")
print("✅ LangSmith tracing enabled")
print("✅ Project: AI-Health-Information-Researcher")

🔐 AI Health Information Researcher
Enter your Gemini API key: ··········
Enter your Tavily API key: ··········
Enter your LangSmith API key: ··········

✅ API keys loaded
✅ LangSmith tracing enabled
✅ Project: AI-Health-Information-Researcher


In [16]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(
    max_results=3
)

print("✅ Tavily configured")

✅ Tavily configured


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

print("✅ Gemini configured")
print("🤖 Model: gemini-3.8-flash")

✅ Gemini configured
🤖 Model: gemini-3.8-flash


In [18]:
from langsmith import Client

langsmith_client = Client(
    api_key=os.environ["LANGSMITH_API_KEY"]
)

print("✅ LangSmith configured")
print("📊 Project: AI-Health-Information-Researcher")

✅ LangSmith configured
📊 Project: AI-Health-Information-Researcher


In [19]:
from typing import TypedDict, List, Dict


class HealthResearchState(TypedDict):
    question: str
    search_results: List[Dict]
    research_text: str
    answer: str
    sources: List[str]


print("✅ HealthResearchState created")

✅ HealthResearchState created


In [20]:
def health_research_node(state: HealthResearchState):

    question = state["question"]

    print("\n🔎 TAVILY WEB RESEARCH")
    print("-" * 60)
    print("Question:", question)

    results = tavily_search.invoke(question)

    research_text = ""
    sources = []

    for i, result in enumerate(
        results.get("results", []),
        start=1
    ):

        title = result.get(
            "title",
            "Unknown"
        )

        url = result.get(
            "url",
            ""
        )

        content = result.get(
            "content",
            ""
        )

        research_text += f"""
SOURCE {i}

TITLE:
{title}

URL:
{url}

CONTENT:
{content}

----------------------------------------
"""

        if url:
            sources.append(url)

    print(
        f"✅ Tavily found {len(sources)} sources"
    )

    return {
        "search_results": results.get(
            "results",
            []
        ),
        "research_text": research_text,
        "sources": sources
    }


print("✅ Tavily research node created")

✅ Tavily research node created


In [21]:
from langchain_core.messages import HumanMessage


def health_analysis_node(state: HealthResearchState):

    question = state["question"]
    research = state["research_text"]

    print("\n🤖 GEMINI AI ANALYSIS")
    print("-" * 60)

    prompt = f"""
You are an AI Health Information Researcher.

Your purpose is to provide GENERAL EDUCATIONAL HEALTH INFORMATION ONLY.

You are NOT a doctor.

SAFETY RULES:

- Do NOT diagnose the user.
- Do NOT claim that the user definitely has a disease.
- Do NOT prescribe medication.
- Do NOT provide medication dosages.
- Do NOT provide personalized treatment plans.
- Do NOT provide personalized medical decisions.
- Do NOT invent medical facts.
- Use only information supported by the supplied web research.
- Clearly explain uncertainty when appropriate.
- Encourage consultation with a qualified healthcare professional
  when appropriate.

USER QUESTION:
{question}

WEB RESEARCH:
{research}

Create a clear educational response.

Use this structure:

## Summary

Give a short explanation.

## Key Information

Explain the important information in simple language.

## Common Signs or Symptoms

List relevant signs or symptoms when applicable.

## Important Considerations

Explain limitations and uncertainty.

## When to Seek Professional Help

Give general guidance about when someone should consider
speaking with a qualified healthcare professional.

## Sources

List the relevant sources from the supplied research.

Return ONLY the final answer.
"""

    try:

        response = gemini_llm.invoke(
            [HumanMessage(content=prompt)]
        )

        answer = response.content

        print("✅ Gemini analysis completed")

        return {
            "answer": answer
        }

    except Exception as e:

        error_message = str(e)

        print("❌ Gemini analysis failed")

        # Handle quota exhaustion
        if (
            "429" in error_message
            or "RESOURCE_EXHAUSTED" in error_message
            or "quota" in error_message.lower()
        ):

            answer = """
## Gemini API Quota Temporarily Exhausted

The web research was completed successfully, but Gemini
could not generate the AI analysis because the Gemini API
quota is currently exhausted.

### Current Status

✅ Tavily web research: Completed

❌ Gemini AI analysis: Temporarily unavailable

Please try again after the Gemini quota becomes available.

---

⚠️ **Educational Information Only**

This application provides general educational health
information. It is not a medical diagnosis and is not
a substitute for advice from a qualified healthcare
professional.
"""

        else:

            answer = f"""
## AI Analysis Temporarily Unavailable

The web research was completed, but the AI analysis
could not be generated because of a temporary service error.

Please try again later.

Technical information:

{error_message}

---

⚠️ **Educational Information Only**

This application provides general educational health
information. It is not a medical diagnosis and is not
a substitute for advice from a qualified healthcare
professional.
"""

        return {
            "answer": answer
        }


print("✅ Gemini analysis node created")

✅ Gemini analysis node created


In [22]:
def health_safety_node(state: HealthResearchState):

    answer = state["answer"]

    print("\n🛡️ SAFETY CHECK")
    print("-" * 60)

    disclaimer = """
---

⚠️ **Educational Information Only**

This information is provided for general educational purposes.
It is not a medical diagnosis and is not a substitute for advice
from a qualified healthcare professional.
"""

    if "Educational Information Only" not in answer:

        answer = answer + disclaimer

    print("✅ Safety check completed")

    return {
        "answer": answer
    }


print("✅ Safety node created")

✅ Safety node created


In [23]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(
    HealthResearchState
)


# ------------------------------------------------------------
# ADD NODES
# ------------------------------------------------------------

builder.add_node(
    "health_research",
    health_research_node
)

builder.add_node(
    "health_analysis",
    health_analysis_node
)

builder.add_node(
    "health_safety",
    health_safety_node
)


# ------------------------------------------------------------
# CONNECT NODES
# ------------------------------------------------------------

builder.add_edge(
    START,
    "health_research"
)

builder.add_edge(
    "health_research",
    "health_analysis"
)

builder.add_edge(
    "health_analysis",
    "health_safety"
)

builder.add_edge(
    "health_safety",
    END
)


# ------------------------------------------------------------
# COMPILE
# ------------------------------------------------------------

health_graph = builder.compile()


print("=" * 70)
print("✅ LANGGRAPH CREATED SUCCESSFULLY")
print("=" * 70)

print("""
START
  ↓
🔎 Tavily Research
  ↓
🤖 Gemini Analysis
  ↓
🛡️ Safety Check
  ↓
END
""")

✅ LANGGRAPH CREATED SUCCESSFULLY

START
  ↓
🔎 Tavily Research
  ↓
🤖 Gemini Analysis
  ↓
🛡️ Safety Check
  ↓
END



In [24]:
import getpass

print("📧 GMAIL CONFIGURATION")
print("=" * 60)

GMAIL_SENDER = input(
    "Enter your Gmail address: "
).strip()

GMAIL_APP_PASSWORD = getpass.getpass(
    "Enter your 16-character Google App Password: "
).strip().replace(" ", "")

GMAIL_RECEIVER = input(
    "Enter the email address where results should be sent: "
).strip()

os.environ["GMAIL_SENDER"] = GMAIL_SENDER
os.environ["GMAIL_APP_PASSWORD"] = GMAIL_APP_PASSWORD
os.environ["GMAIL_RECEIVER"] = GMAIL_RECEIVER

print("\n✅ Gmail configuration loaded")
print("📧 Sender:", GMAIL_SENDER)
print("📨 Receiver:", GMAIL_RECEIVER)
print("🔐 App password stored securely")

📧 GMAIL CONFIGURATION
Enter your Gmail address: yoginikumar0608@gmail.com
Enter your 16-character Google App Password: ··········
Enter the email address where results should be sent: yoginikumar0608@gmail.com

✅ Gmail configuration loaded
📧 Sender: yoginikumar0608@gmail.com
📨 Receiver: yoginikumar0608@gmail.com
🔐 App password stored securely


In [25]:
import smtplib

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from html import escape


def send_health_result_email(
    question,
    answer,
    sources
):

    sender = os.environ["GMAIL_SENDER"]
    app_password = os.environ["GMAIL_APP_PASSWORD"]
    receiver = os.environ["GMAIL_RECEIVER"]

    subject = "🩺 AI Health Information Research Result"

    # --------------------------------------------------------
    # QUESTION
    # --------------------------------------------------------

    safe_question = escape(
        str(question)
    )

    # --------------------------------------------------------
    # ANSWER
    # --------------------------------------------------------

    safe_answer = escape(
        str(answer)
    ).replace(
        "\n",
        "<br>"
    )

    # --------------------------------------------------------
    # SOURCES
    # --------------------------------------------------------

    source_html = ""

    if sources:

        for i, source in enumerate(
            sources,
            start=1
        ):

            safe_source = escape(
                str(source)
            )

            source_html += f"""
            <li>
                <a href="{safe_source}">
                    {safe_source}
                </a>
            </li>
            """

    else:

        source_html = """
        <li>No sources available.</li>
        """

    # --------------------------------------------------------
    # EMAIL HTML
    # --------------------------------------------------------

    html_content = f"""
    <!DOCTYPE html>

    <html>

    <body style="
        font-family: Arial, sans-serif;
        line-height: 1.6;
        color: #222;
    ">

        <h1>
            🩺 AI Health Information Researcher
        </h1>

        <hr>

        <h2>
            🔍 Question
        </h2>

        <p>
            {safe_question}
        </p>

        <h2>
            🩺 Research Result
        </h2>

        <div>
            {safe_answer}
        </div>

        <h2>
            🔗 Research Sources
        </h2>

        <ol>
            {source_html}
        </ol>

        <hr>

        <h3>
            ⚠️ Educational Information Only
        </h3>

        <p>
            This information is provided for general educational
            purposes. It is not a medical diagnosis and is not a
            substitute for advice from a qualified healthcare
            professional.
        </p>

        <p>
            Generated by AI Health Information Researcher.
        </p>

    </body>

    </html>
    """

    # --------------------------------------------------------
    # CREATE EMAIL
    # --------------------------------------------------------

    message = MIMEMultipart(
        "alternative"
    )

    message["From"] = sender
    message["To"] = receiver
    message["Subject"] = subject

    message.attach(
        MIMEText(
            html_content,
            "html"
        )
    )

    # --------------------------------------------------------
    # SEND EMAIL
    # --------------------------------------------------------

    try:

        with smtplib.SMTP(
            "smtp.gmail.com",
            587
        ) as server:

            server.ehlo()
            server.starttls()
            server.ehlo()

            server.login(
                sender,
                app_password
            )

            server.sendmail(
                sender,
                receiver,
                message.as_string()
            )

        print("📧 Email sent successfully!")

        return True

    except Exception as e:

        print("❌ Email sending failed")
        print("Error:", str(e))

        return False


print("✅ Gmail function created")

✅ Gmail function created


In [26]:
from langsmith import traceable


@traceable(
    name="AI Health Research Request"
)
def research_health_question(question):

    # --------------------------------------------------------
    # VALIDATE QUESTION
    # --------------------------------------------------------

    if not question or not question.strip():

        return (
            "⚠️ **Please enter a health-related question.**",
            "No sources available.",
            "❌ Email not sent."
        )

    question = question.strip()

    print("\n" + "=" * 70)
    print("🩺 AI HEALTH INFORMATION RESEARCHER")
    print("=" * 70)

    print("Question:", question)

    # --------------------------------------------------------
    # INITIAL LANGGRAPH STATE
    # --------------------------------------------------------

    initial_state = {

        "question": question,

        "search_results": [],

        "research_text": "",

        "answer": "",

        "sources": []
    }

    try:

        # ----------------------------------------------------
        # RUN LANGGRAPH
        # ----------------------------------------------------

        result = health_graph.invoke(
            initial_state
        )

        # ----------------------------------------------------
        # GET RESULT
        # ----------------------------------------------------

        answer = result.get(
            "answer",
            "No answer generated."
        )

        sources = result.get(
            "sources",
            []
        )

        # ----------------------------------------------------
        # FORMAT SOURCES
        # ----------------------------------------------------

        if sources:

            sources_text = "\n\n".join(
                f"**{i}.** {source}"
                for i, source in enumerate(
                    sources,
                    start=1
                )
            )

        else:

            sources_text = (
                "No sources were returned."
            )

        # ----------------------------------------------------
        # SEND EMAIL
        # ----------------------------------------------------

        print("\n📧 Sending result to Gmail...")

        email_sent = send_health_result_email(
            question=question,
            answer=answer,
            sources=sources
        )

        if email_sent:

            email_status = (
                "📧 **Result successfully sent to Gmail.**"
            )

        else:

            email_status = (
                "⚠️ **Research completed, "
                "but email sending failed.**"
            )

        print("\n✅ Research request completed")

        return (
            answer,
            sources_text,
            email_status
        )

    except Exception as e:

        print("\n❌ Research request failed")
        print("Error:", str(e))

        return (
            """
## ⚠️ Service Temporarily Unavailable

The health research service could not complete this request.

Please try again later.

---

⚠️ **Educational Information Only**

This application provides general educational health
information and is not a substitute for professional
medical advice.
""",
            "Research sources unavailable.",
            "❌ Email not sent because the research failed."
        )


print("✅ Main research backend created")

✅ Main research backend created


In [27]:
import gradio as gr


def run_research(question):

    answer, sources, email_status = (
        research_health_question(question)
    )

    return (
        "✅ Research completed",
        answer,
        sources,
        email_status
    )


def clear_all():

    return (
        "",
        "",
        "",
        "",
        ""
    )


with gr.Blocks(
    title="AI Health Information Researcher",
    theme=gr.themes.Soft()
) as app:

    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
# 🩺 AI Health Information Researcher

### Research health information using AI + Web Search

**Tavily 🔎 · Gemini 🤖 · LangGraph 🧠 ·
LangSmith 📊 · Gmail 📧**
"""
    )

    # ========================================================
    # SAFETY NOTICE
    # ========================================================

    gr.Markdown(
        """
> ⚠️ **Educational Information Only**
>
> This tool provides general educational health information.
> It does **not diagnose diseases, prescribe medication,
> or replace professional medical advice.**
"""
    )

    # ========================================================
    # INPUT
    # ========================================================

    gr.Markdown(
        "## 🔍 Ask a Health Question"
    )

    question_input = gr.Textbox(
        label="Your Question",
        placeholder=(
            "Example: What is hypertension?"
        ),
        lines=4
    )

    # ========================================================
    # BUTTONS
    # ========================================================

    with gr.Row():

        research_button = gr.Button(
            "🔎 Research",
            variant="primary",
            size="lg"
        )

        clear_button = gr.Button(
            "🧹 Clear",
            size="lg"
        )

    # ========================================================
    # STATUS
    # ========================================================

    status = gr.Markdown()

    # ========================================================
    # RESULT
    # ========================================================

    gr.Markdown(
        "## 🩺 AI Research Result"
    )

    answer_output = gr.Markdown()

    # ========================================================
    # SOURCES
    # ========================================================

    gr.Markdown(
        "## 🔗 Research Sources"
    )

    sources_output = gr.Markdown()

    # ========================================================
    # EMAIL
    # ========================================================

    gr.Markdown(
        "## 📧 Email Delivery"
    )

    email_status_output = gr.Markdown()

    # ========================================================
    # EXAMPLES
    # ========================================================

    gr.Markdown(
        "## 💡 Try an Example"
    )

    gr.Examples(
        examples=[
            [
                "What is hypertension?"
            ],
            [
                "What are the common symptoms of iron deficiency?"
            ],
            [
                "What are common symptoms of vitamin D deficiency?"
            ],
            [
                "What are common causes of dehydration?"
            ],
            [
                "What are common symptoms of seasonal allergies?"
            ]
        ],
        inputs=question_input
    )

    # ========================================================
    # RESEARCH BUTTON EVENT
    # ========================================================

    research_button.click(
        fn=run_research,

        inputs=question_input,

        outputs=[
            status,
            answer_output,
            sources_output,
            email_status_output
        ]
    )

    # ========================================================
    # CLEAR BUTTON EVENT
    # ========================================================

    clear_button.click(
        fn=clear_all,

        inputs=None,

        outputs=[
            question_input,
            status,
            answer_output,
            sources_output,
            email_status_output
        ]
    )

    # ========================================================
    # HOW IT WORKS
    # ========================================================

    gr.Markdown(
        """
---

## 🧠 How This AI Works

**1️⃣ Question**

You enter a health-related question.

**2️⃣ Web Research**

Tavily searches the web for relevant information.

**3️⃣ AI Analysis**

Gemini analyzes the collected research.

**4️⃣ Safety Guardrail**

A local Python safety layer adds an educational-use disclaimer.

**5️⃣ Sources**

The application displays the sources used during research.

**6️⃣ Gmail Delivery**

The result is automatically sent to the configured Gmail address.

**7️⃣ LangSmith**

LangSmith traces the workflow for monitoring and debugging.

---

### 🛠️ Technologies

`Python` · `LangChain` · `LangGraph` · `Gemini` ·
`Tavily` · `LangSmith` · `Gradio` · `Gmail SMTP`
"""
    )


print("✅ Gradio application created successfully")

/tmp/ipykernel_22006/298966677.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio application created successfully


In [29]:
app.launch(
    share=True,
    debug=False
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14f64549d634fa5708.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
